# ⚡ Revio Intelligence Engine
## Aspect-Based Sentiment Mining, Churn/Crisis Early Warning & Autonomous Response Engine

**Engine Version:** v1.0.0 Enterprise  
**Architecture:** Multi-Tiered Hybrid NLP (ABSA + Heuristic Risk Scorer + Guardrailed GenAI)  

---
### 📌 Executive Overview
Modern enterprises operating multi-channel products often struggle to extract actionable operational intelligence from high volumes of customer feedback across Google, Trustpilot, G2, and App Store.

While current review platforms offer basic star tracking and generic auto-replies, businesses lose revenue due to:
1. **Superficial Sentiment Analysis:** Standard positive/negative tags miss granular product/operational flaws.
2. **Uncaught Crisis & Churn Escalations:** High-risk customer issues (server outages, billing disputes, data leaks) sit in support queues until public damage is done.
3. **Generic Robotic Responses:** Automated replies often sound uninspired, repetitious, or worse, make legally unauthorized promises.

**The Revio-Intelligence Solution:**
- **Aspect-Based Sentiment Analysis (ABSA)** across 5 core dimensions (*Customer Support*, *Pricing & Billing*, *System Reliability*, *Product Features*, *Ease of Use*).
- **Probabilistic Urgency & Crisis Scorer (0-100)** with real-time churn threat detection and SLA routing.
- **Brand-Aligned Response Synthesis** with strict safety guardrails and Human-in-the-Loop gates.

### 🛠️ 1. Setup & Environment Initialization

In [ ]:
import re
import math
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional
import pandas as pd

print("Environment initialized successfully!")

### 📊 2. Omnichannel Benchmark Dataset
Realistic benchmark dataset reflecting customer reviews across G2, Trustpilot, Google Reviews, and App Store.

In [ ]:
SAMPLE_REVIEWS = [
    {"review_id": "REV-1001", "platform": "G2", "author": "Sarah Jenkins", "rating": 5, "text": "Revio completely streamlined our multi-location Google review monitoring. The automated alerts are instantaneous and our team responded 4x faster."},
    {"review_id": "REV-1002", "platform": "Trustpilot", "author": "Marcus Vance", "rating": 1, "text": "Unacceptable server outage during our holiday marketing campaign. Your dashboard was unreachable for 4 hours and support took 2 days to reply. We are considering terminating our enterprise contract immediately."},
    {"review_id": "REV-1003", "platform": "Google Reviews", "author": "Priya Sharma", "rating": 2, "text": "The maintenance ticket feature in TenantEase is full of bugs. I logged an urgent water leakage 3 days ago, and the landlord app showed no notification. Fix your notification delivery!"},
    {"review_id": "REV-1004", "platform": "App Store", "author": "David Miller", "rating": 5, "text": "TenantEase made rent payments and lease renewal seamless. Love the automated rent reminders and receipt generation. Best property app we've used."},
    {"review_id": "REV-1005", "platform": "Clutch", "author": "Elena Rostova", "rating": 5, "text": "The engineering team developed our custom AI agent workflow within 6 weeks. The software quality, AWS cloud architecture, and communication were world-class."},
    {"review_id": "REV-1006", "platform": "Google Reviews", "author": "Rajiv Mehta", "rating": 3, "text": "The web app delivered was technically solid and fast, but the initial onboarding and milestone documentation were delayed. Pricing is slightly on the higher side."},
    {"review_id": "REV-1007", "platform": "Trustpilot", "author": "Amir Khan", "rating": 1, "text": "Total scam. I was double-charged for my October maintenance fee and your payment gateway showed failed transaction. I want an immediate refund or I will file a bank chargeback."}
]

df_samples = pd.DataFrame(SAMPLE_REVIEWS)
df_samples

### 🧠 3. Core ABSA & Crisis Engine Architecture
Implements sentence tokenization, clause contrastive splitting, aspect extraction, and risk heuristic calculation.

In [ ]:
ASPECT_TAXONOMY = {
    "Customer Support": {"keywords": ["support", "agent", "ticket", "service", "helpdesk", "reply", "response time"], "dept": "Customer Success & Support"},
    "Pricing & Billing": {"keywords": ["price", "pricing", "cost", "expensive", "bill", "fee", "charge", "double-charged", "refund", "chargeback"], "dept": "Finance & Billing"},
    "System Reliability": {"keywords": ["outage", "downtime", "crash", "bug", "slow", "server", "down", "uptime", "leak", "security"], "dept": "Core Infrastructure & DevOps"},
    "Product Features": {"keywords": ["feature", "ui", "ux", "dashboard", "tool", "workflow", "automation", "agent", "app"], "dept": "Product & Engineering"},
    "Ease of Use": {"keywords": ["easy", "intuitive", "smooth", "seamless", "clean", "simple", "complicated", "confusing"], "dept": "Design & UX"}
}

CRISIS_TRIGGERS = [
    (r"\b(security|leak|leaked|breach|vulnerability|credentials)\b", 40, "Security Threat"),
    (r"\b(scam|fraud|lawsuit|legal|sue|court|regulator)\b", 45, "Legal / Fraud Risk"),
    (r"\b(chargeback|double-charged|dispute|refund)\b", 35, "Billing Dispute"),
    (r"\b(terminate|cancel contract|churn|switching to|leaving)\b", 35, "High Churn Risk"),
    (r"\b(outage|unreachable|down for \d+|lost data)\b", 30, "Severe Outage"),
    (r"\b(unacceptable|horrible|worst|disaster|furious)\b", 20, "Severe Dissatisfaction")
]

POSITIVE_WORDS = {"great", "excellent", "outstanding", "seamless", "fast", "love", "best", "top-notch", "reliable", "helpful", "good", "amazing", "smooth", "world-class"}
NEGATIVE_WORDS = {"bad", "terrible", "worst", "unacceptable", "slow", "bug", "bugs", "outage", "downtime", "delayed", "expensive", "scam", "fraud", "leak", "broken", "double-charged"}

def score_clause_sentiment(text: str) -> float:
    words = re.findall(r"\b[\w'-]+\b", text.lower())
    if not words:
        return 0.0
    val = sum(1.0 if w in POSITIVE_WORDS else (-1.2 if w in NEGATIVE_WORDS else 0.0) for w in words)
    return round(math.tanh(val / 2.5), 3)

def analyze_review_demo(text: str, rating: Optional[int] = None):
    clauses = re.split(r'[,;]|\s+but\s+|\s+however\s+', text, flags=re.IGNORECASE)
    detected_aspects = []
    
    for aspect_name, data in ASPECT_TAXONOMY.items():
        for c in clauses:
            if any(re.search(rf"\b{re.escape(k)}\b", c.lower()) for k in data["keywords"]):
                sc = score_clause_sentiment(c)
                sentiment = "Positive" if sc > 0.1 else ("Negative" if sc < -0.1 else "Neutral")
                detected_aspects.append({"aspect": aspect_name, "sentiment": sentiment, "score": sc, "dept": data["dept"]})
                break
                
    # Crisis & Urgency
    urgency = 40 if rating == 1 else (25 if rating == 2 else 5)
    tags = []
    for pat, weight, tag in CRISIS_TRIGGERS:
        if re.search(pat, text.lower()):
            urgency += weight
            tags.append(tag)
            
    urgency = min(100, urgency)
    is_crisis = urgency >= 65 or any("Security" in t or "Legal" in t for t in tags)
    churn_risk = "High" if (urgency >= 70 or "High Churn Risk" in tags) else ("Medium" if urgency >= 40 else "Low")
    
    return {
        "aspects": detected_aspects,
        "urgency": urgency,
        "is_crisis": is_crisis,
        "tags": tags,
        "churn_risk": churn_risk
    }

print("Engine definition complete.")

### ⚡ 4. Live Test on Critical Review

In [ ]:
test_review = "Unacceptable server outage during our holiday marketing campaign. Your dashboard was unreachable for 4 hours and support took 2 days to reply. We are considering terminating our enterprise contract immediately."
result = analyze_review_demo(test_review, rating=1)

print(f"Urgency Index: {result['urgency']}/100 | Is Crisis: {result['is_crisis']}")
print(f"Crisis Tags: {result['tags']}")
print(f"Churn Risk: {result['churn_risk']}")
print("\nDetected Operational Aspects:")
for a in result['aspects']:
    print(f" - {a['aspect']}: {a['sentiment']} (Dept: {a['dept']})")

### 🤖 5. Autonomous Brand Response Synthesis with Safety Guardrails

In [ ]:
def synthesize_brand_response(author: str, analysis: dict) -> dict:
    if analysis["is_crisis"]:
        tone = "Executive Priority Escalation"
        reply = (
            f"Hello {author},\n\n"
            "Thank you for bringing this critical matter to our immediate attention. We take this feedback with the utmost "
            "gravity, and our leadership team has initiated an expedited internal review.\n\n"
            "Please contact our executive escalation team directly at priority@revio-intel.ai so we can resolve this without delay.\n\n"
            "Warm regards,\nThe Customer Experience Team"
        )
        gate = True
    else:
        tone = "Empathetic & Solution-Oriented"
        reply = (
            f"Hi {author},\n\n"
            "Thank you for sharing your feedback. We continuously refine our services based on customer input.\n\n"
            "Best regards,\nThe Customer Support Team"
        )
        gate = False
        
    return {
        "tone": tone,
        "reply": reply,
        "human_review_required": gate
    }

resp = synthesize_brand_response("Marcus Vance", result)
print(f"Tone: {resp['tone']}")
print(f"Human Gate Active: {resp['human_review_required']}\n")
print(resp['reply'])

### 📈 6. Evaluation & Batch Processing Metrics

In [ ]:
batch_records = []
for r in SAMPLE_REVIEWS:
    res = analyze_review_demo(r["text"], r["rating"])
    resp = synthesize_brand_response(r["author"], res)
    batch_records.append({
        "ID": r["review_id"],
        "Platform": r["platform"],
        "Author": r["author"],
        "Rating": r["rating"],
        "Urgency": res["urgency"],
        "Crisis": "YES" if res["is_crisis"] else "No",
        "Churn Risk": res["churn_risk"],
        "Response Tone": resp["tone"],
        "Human Review Gate": resp["human_review_required"]
    })

df_out = pd.DataFrame(batch_records)
print("=== BATCH PROCESSING SUMMARY ===")
df_out

### 🎯 Conclusion & Deployment Architecture
- **FastAPI Backend:** Containerized deployment via Docker/ECS behind an Application Load Balancer.
- **Enterprise SaaS Integration:** Replaces legacy star ratings with real-time churn risk indicators.
- **Automated Escalation:** Instantly prioritizes urgent operational tickets before public reputation damage.